In [14]:
import os
import time
import random

import pandas as pd

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [15]:
driver = webdriver.Chrome()

In [19]:
html = driver.page_source

soup = BeautifulSoup(html,"html.parser")

In [25]:
def ambil_text_multi(soup, selectors):
    for selector in selectors:
        hasil = ambil_text(
            soup,
            tag=selector["tag"],
            class_name=selector.get("class_name"),
            data_test=selector.get("data_test")
        )

        if hasil:
            return hasil

    return None

In [27]:
def scrape_detail(driver, url):

    driver.get(url)

    WebDriverWait(driver,30).until(
        EC.presence_of_element_located((By.TAG_NAME,"h1"))
    )

    soup = BeautifulSoup(driver.page_source,"html.parser")

    # Judul
    judul = ambil_text(
        soup,
        tag="h1"
    )

    # Harga
    harga = ambil_text(
        soup,
        tag="div",
        class_name="prices-and-fees__price",
        data_test="listing-price"
    )

    # Lokasi
    lokasi = soup.find("div", class_="view-map__text")
    lokasi = lokasi.get_text(strip=True) if lokasi else None

    # Kota & Provinsi
    bagian = [x.strip() for x in lokasi.split(",")] if lokasi else []

    kota = bagian[-2] if len(bagian) >= 2 else None
    provinsi = bagian[-1] if len(bagian) >= 1 else None

    # Tipe Properti
    tipe = ambil_text(
        soup,
        tag="span",
        class_name="features-component__value",
        data_test="property-type-value"
    )

    # Kamar Tidur
    kamar_tidur = ambil_text(
        soup,
        tag="div",
        class_name="details-item-value",
        data_test="bedrooms-value"
    )

    # Kamar Mandi
    kamar_mandi = ambil_text(
        soup,
        tag="div",
        class_name="details-item-value",
        data_test="full-bathrooms-value"
    )

    # Luas Total
    luas_total = ambil_text_multi(
        soup,
        [
            {
                "tag": "span",
                "class_name": "features-component__value",
                "data_test": "floor-area-value"
            },
            {
                "tag": "div",
                "class_name": "details-item-value",
                "data_test": "area-value"
            }
        ]
    )

    lantai = ambil_text(
        soup,
        tag="span",
        class_name="features-component__value",
        data_test="floor-value"
    )

    # Tahun Dibangun
    tahun_dibangun = ambil_text(
        soup,
        tag="span",
        class_name="features-component__value",
        data_test="construction-year-value"
    )

    # Sertifikat
    sertifikat = ambil_text(
        soup,
        tag="span",
        class_name="place-features__values",
        data_test="ownership-certificate-value"
    )   

    # Nama Agen
    nama_agen = ambil_text(
        soup,
        tag="span",
        class_name="agency__name",
        data_test="agency-name"
    )

    # Tanggal Posting
    tanggal = soup.find("div", class_="date")
    tanggal = tanggal.get_text(" ", strip=True) if tanggal else None

    if tanggal:
        tanggal = tanggal.split("-")[0].strip()

    return {
        "Judul": judul,
        "Harga": harga,
        "Lokasi": lokasi,
        "Kota": kota,
        "Provinsi": provinsi,
        "Tipe Properti": tipe,
        "Kamar Tidur": kamar_tidur,
        "Kamar Mandi": kamar_mandi,
        "Luas Total": luas_total,
        "Lantai": lantai,
        "Tahun Dibangun": tahun_dibangun,
        "Sertifikat": sertifikat,
        "Nama Agen": nama_agen,
        "Tanggal Posting": tanggal,
        "Link": url
    }

In [24]:
driver = webdriver.Chrome()

data = scrape_detail(
    driver,
    "https://www.lamudi.co.id/properti/41032-73-95cd83cc7304-dbac-19e4a5d-8b75-73a2"
)

driver.quit()

data

NameError: name 'ambil_text' is not defined

In [10]:
import pandas as pd

df = pd.read_csv("data/lamudi_apartemen_links.csv")
print(df.shape)
df.head()

(2100, 4)


,Judul,Harga,Lokasi,Link
0,Apartemen Dijual di Pancoran,Rp 600Jt,"RW 07, Rawajati, Pancoran, Jakarta Selatan, Da...",https://www.lamudi.co.id/properti/41032-73-54c...
1,Apartemen Dijual di Bumi Serpong Damai,Rp 860Jt,"BSD City, Pagedangan, Pagedangan, Kabupaten Ta...",https://www.lamudi.co.id/properti/41032-73-95c...
2,Apartemen Dijual di Bumi Serpong Damai,Rp 850Jt,"BSD City, Pagedangan, Pagedangan, Kabupaten Ta...",https://www.lamudi.co.id/properti/41032-73-6b3...
3,Apartemen Dijual di Bumi Serpong Damai,Rp 840Jt,"BSD Grand CBD, BSD City, Pagedangan, Pagedanga...",https://www.lamudi.co.id/properti/41032-73-246...
4,Apartemen Dijual di Balikpapan Selatan,Rp 600Jt,"Sepinggan Baru, Balikpapan Selatan, Balikpapan...",https://www.lamudi.co.id/properti/41032-73-f74...


In [11]:
TOTAL_DATA = 10

driver = webdriver.Chrome()

driver.set_page_load_timeout(30)

hasil = []

start_time = time.time()

In [12]:
for i, url in enumerate(df["Link"].head(TOTAL_DATA), start=1):

    try:

        print(f"[{i}/{TOTAL_DATA}]")

        data = scrape_detail(driver, url)

        hasil.append(data)

        # Autosave setiap 50 data
        if i % 50 == 0:

            hasil_df = pd.DataFrame(hasil)

            hasil_df.to_csv(
                "data/lamudi_apartemen_detail_temp.csv",
                index=False,
                encoding="utf-8-sig"
            )

            elapsed = time.time() - start_time
            avg = elapsed / i
            remaining = (TOTAL_DATA - i) * avg

            print("=" * 60)
            print(f"Autosave {i} data")
            print(f"Waktu berjalan : {elapsed/60:.2f} menit")
            print(f"Rata-rata : {avg:.2f} detik/data")
            print(f"Estimasi sisa : {remaining/60:.2f} menit")
            print("=" * 60)

        time.sleep(random.uniform(1,2))

    except Exception as e:

        print("=" * 60)
        print(f"Gagal data ke-{i}")
        print(url)
        print(e)
        print("=" * 60)

driver.quit()

print("Browser ditutup")

[1/10]
Gagal data ke-1
https://www.lamudi.co.id/properti/41032-73-54ce27f8200-7466-19e91bd-aaf4-73c3
name 'ambil_text' is not defined
[2/10]
Gagal data ke-2
https://www.lamudi.co.id/properti/41032-73-95cd83cc7304-dbac-19e4a5d-8b75-73a2
name 'ambil_text' is not defined
[3/10]
Gagal data ke-3
https://www.lamudi.co.id/properti/41032-73-6b37fd621feb-bba4-19e881f-983d-8029
name 'ambil_text' is not defined
[4/10]
Gagal data ke-4
https://www.lamudi.co.id/properti/41032-73-2462c22c8a4b-cf35-19e8efa-b9dd-7c00
name 'ambil_text' is not defined
[5/10]
Gagal data ke-5
https://www.lamudi.co.id/properti/41032-73-f74d68fda560-862d-19a6d33-89c7-734b
name 'ambil_text' is not defined
[6/10]
Gagal data ke-6
https://www.lamudi.co.id/properti/41032-73-d6cd955edaef-3f5a-19b7d15-b9f3-7ef6
name 'ambil_text' is not defined
[7/10]
Gagal data ke-7
https://www.lamudi.co.id/properti/41032-73-49f11b01286d-ca91-19e9ca6-85c4-7df1
name 'ambil_text' is not defined
[8/10]
Gagal data ke-8
https://www.lamudi.co.id/properti

In [ ]:
hasil_df = pd.DataFrame(hasil)

print(hasil_df.shape)

hasil_df.head()

(10, 15)


,Judul,Harga,Lokasi,Kota,Provinsi,Tipe Properti,Kamar Tidur,Kamar Mandi,Luas Total,Lantai,Tahun Dibangun,Sertifikat,Nama Agen,Tanggal Posting,Link
0,DIJUAL CEPAT UNIT STUDIO TOWER MATOA LT. 17 WO...,Rp 600Jt,"RW 07, Rawajati, Pancoran, Jakarta Selatan, Da...",Jakarta Selatan,Daerah Khusus Ibukota Jakarta,Apartemen,1 kamar tidur,1 kamar mandi,28 m²,17,2015,PPJB,MoFu,4 Jun 2026,https://www.lamudi.co.id/properti/41032-73-54c...
1,DI JUAL APARTEMEN 2 BEDROOM SKY HOUSE BSD – RE...,Rp 860Jt,"BSD City, Pagedangan, Pagedangan, Kabupaten Ta...",Kabupaten Tangerang,Banten,Apartemen,2 Kamar Tidur,1 kamar mandi,51 m²,19,2023,SHM,Frans Property,21 Mei 2026,https://www.lamudi.co.id/properti/41032-73-95c...
2,"Di jual 2BR Sky House BSD city, lokasi premium...",Rp 850Jt,"BSD City, Pagedangan, Pagedangan, Kabupaten Ta...",Kabupaten Tangerang,Banten,Apartemen,2 Kamar Tidur,1 kamar mandi,51 m²,20,2023,SHM,Frans Property,2 Jun 2026,https://www.lamudi.co.id/properti/41032-73-6b3...
3,INVESTASI TERBAIK 2026: 2BR SAMPING AEON BSD H...,Rp 840Jt,"BSD Grand CBD, BSD City, Pagedangan, Pagedanga...",Kabupaten Tangerang,Banten,Apartemen,2 Kamar Tidur,1 kamar mandi,48 m²,26,2027,SHM,Sky House BSD.,3 Jun 2026,https://www.lamudi.co.id/properti/41032-73-246...
4,Dijual Apartemen Sepinggan Terrace Balikpapan,Rp 600Jt,"Sepinggan Baru, Balikpapan Selatan, Balikpapan...",Balikpapan,Kalimantan Timur,Apartemen,NaN,1 kamar mandi,29 m²,NaN,NaN,NaN,Frans Property,10 Nov 2025,https://www.lamudi.co.id/properti/41032-73-f74...


In [ ]:
hasil_df.to_csv(
    "data/lamudi_apartemen_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Berhasil disimpan")

In [ ]:
hasil_df.info()

hasil_df.isnull().sum()